# Insurance Policy Comparison Agent — Live Analysis

Run cells **in order** (top to bottom).

**Before starting:**
- `cd ~/capstone && source .venv/bin/activate && jupyter notebook`
- Ollama app running **or** Groq configured in `.env` (see cell 1 output)
- Cell 3 can take **several minutes** — wait for `[*]` to finish

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langgraph")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path("/Users/preetidave/capstone")
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

from src.config import LLM_PROVIDER, get_llm_mode_label

print("Working directory:", os.getcwd())
print("LLM provider:", LLM_PROVIDER)
print("Model label:", get_llm_mode_label())
print("Ready.", flush=True)

In [ ]:
# Quick LLM ping — should finish in a few seconds
from src.llm.client import complete

print("Testing one LLM call...", flush=True)
reply = complete("You are a test assistant.", "Reply with exactly: ok", temperature=0)
print("LLM replied:", repr(reply), flush=True)

In [ ]:
from IPython.display import Markdown, display
from src.runner import run_agent, build_user_messages

policies = [
    str(ROOT / "data/synthetic/plan_a.txt"),
    str(ROOT / "data/synthetic/plan_b.txt"),
]
messages = build_user_messages(
    age=35,
    location="Cedar Park, TX",
    flood_zone=True,
    coverage_breadth=0.4,
    low_cost=0.3,
    few_exclusions=0.3,
)

print("Starting full agent workflow (intake → index → ToT → synthesis)...", flush=True)
print("Live inference may take several minutes. Wait until [*] becomes a number.", flush=True)

result = run_agent(
    policy_paths=policies,
    user_messages=messages,
    beam_width=2,
    max_depth=2,
)

print("\n--- DONE ---", flush=True)
print("Mode:", result.get("mode"))
print("LLM calls:", result.get("llm_call_count"))
if result.get("error"):
    print("Error:", result["error"])
display(Markdown(result.get("final_recommendation") or "_No recommendation generated._"))

In [ ]:
# Optional: public HO-3 specimen policies instead of synthetic demo
public_policies = [
    str(ROOT / "data/public/travelers_ho3_nv.txt"),
    str(ROOT / "data/public/statefarm_hw2136_ok.txt"),
]

print("Running with public HO-3 specimens...", flush=True)
result_public = run_agent(
    policy_paths=public_policies,
    user_messages=messages,
    beam_width=2,
    max_depth=2,
)

print("Mode:", result_public.get("mode"))
print("LLM calls:", result_public.get("llm_call_count"))
display(Markdown(result_public.get("final_recommendation") or "_No output._"))